# 🧠 Step 9 — Dual TCN (Short-Term + Long-Term Parallel Branches)

## Motivation

Markets are driven by patterns at **different time scales simultaneously**:
- **Short-term (5 days):** Recent momentum, mean reversion, immediate sentiment impact
- **Long-term (20 days):** Trend direction, support/resistance levels, macro cycle

A single TCN with one sequence length captures only one time scale.
The Dual TCN runs **two parallel TCN branches** and merges their outputs:

```
Last 5 days  → Short TCN (dilations 1,2)    → 32-dim short-term signal ──┐
Last 20 days → Long  TCN (dilations 1,2,4,8)→ 32-dim long-term signal  ──┤
FinBERT+Sent → Text Projection              → 64-dim text signal        ──┤
                                                                          ↓
                                            Concat (128-dim) → Deep FNN → prediction
```

## Key difference from single TCN
- Short branch sees **last 5 days only** — captures RSI momentum, MACD crossover
- Long branch sees **last 20 days** — captures trend, Bollinger band position
- Both are simpler than cross-attention so less overfitting risk

## Files needed (same 5)
1. `nifty50_step4_standard_train.csv`
2. `nifty50_step4_standard_test.csv`
3. `step7b_finbert_embeddings.csv`
4. `step7a_finbert_sentiment_comparison.csv`
5. `standard_scaler.pkl`

---
**Enable GPU:** Runtime → Change runtime type → T4 GPU

Then: **Runtime → Run All**

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 1 — Install Libraries
# ─────────────────────────────────────────────────────────────────────────
!pip install torch pandas numpy scikit-learn joblib -q
print('✓ Libraries ready')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 2 — Imports and Configuration
# ─────────────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import joblib
import os
import math
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✓ Device: {device}')
if str(device) == 'cpu':
    print('  ⚠️  No GPU — enable via Runtime → Change runtime type → T4 GPU')

# ── MODEL CONFIG ──────────────────────────────────────────────────────────
SHORT_SEQ_LEN    = 5      # short branch — last 5 days (recent momentum)
LONG_SEQ_LEN     = 20     # long branch  — last 20 days (trend)
TCN_CHANNELS     = 32     # channels per branch (32 each = 64 total when merged)
TCN_KERNEL_SIZE  = 3
SHORT_DILATIONS  = [1, 2]           # short branch — local patterns only
LONG_DILATIONS   = [1, 2, 4, 8]    # long branch  — multi-scale patterns
TCN_DROPOUT      = 0.2
TEXT_PROJ_DIM    = 64
FUSION_DIMS      = [256, 128, 64, 32]
DROPOUT          = 0.2
LEARNING_RATE    = 3e-5
EPOCHS           = 200
BATCH_SIZE       = 16
PATIENCE         = 25
HUBER_DELTA      = 0.5
WARMUP_EPOCHS    = 10

print(f'\n  Architecture: Dual TCN (Short + Long) + Deep Fusion FNN')
print(f'  Short branch : seq_len={SHORT_SEQ_LEN}  dilations={SHORT_DILATIONS}  channels={TCN_CHANNELS}')
print(f'  Long  branch : seq_len={LONG_SEQ_LEN}  dilations={LONG_DILATIONS}  channels={TCN_CHANNELS}')
print(f'  Text proj    : {TEXT_PROJ_DIM}-dim')
print(f'  Fusion FNN   : {TCN_CHANNELS*2}+{TEXT_PROJ_DIM}={TCN_CHANNELS*2+TEXT_PROJ_DIM} → {" → ".join(map(str,FUSION_DIMS))} → 1')
print(f'  Loss         : Huber (delta={HUBER_DELTA})')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 3 — Upload and Load All 5 Input Files
# ─────────────────────────────────────────────────────────────────────────

print('Upload 1/5: nifty50_step4_standard_train.csv')
u = files.upload(); train_num_df = pd.read_csv(list(u.keys())[0])
print(f'  ✓ {len(train_num_df)} train rows')

print('\nUpload 2/5: nifty50_step4_standard_test.csv')
u = files.upload(); test_num_df = pd.read_csv(list(u.keys())[0])
print(f'  ✓ {len(test_num_df)} test rows')

print('\nUpload 3/5: step7b_finbert_embeddings.csv')
u = files.upload(); emb_df = pd.read_csv(list(u.keys())[0])
print(f'  ✓ {len(emb_df)} embedding rows')

print('\nUpload 4/5: step7a_finbert_sentiment_comparison.csv')
u = files.upload(); sent_df = pd.read_csv(list(u.keys())[0])
print(f'  ✓ {len(sent_df)} sentiment rows')

print('\nUpload 5/5: standard_scaler.pkl')
u = files.upload(); scaler = joblib.load(list(u.keys())[0])
print(f'  ✓ Scaler loaded')

# ── Parse dates ────────────────────────────────────────────────────────────
for df in [train_num_df, test_num_df]:
    df['date'] = pd.to_datetime(
        df['Date'], dayfirst=True, errors='coerce'
    ).dt.strftime('%Y-%m-%d')

emb_df['date'] = pd.to_datetime(
    emb_df['date'], dayfirst=True, errors='coerce'
).dt.strftime('%Y-%m-%d')

sent_df['date'] = pd.to_datetime(
    sent_df['date'], dayfirst=True, errors='coerce'
).dt.strftime('%Y-%m-%d')
sent_df = sent_df.dropna(subset=['date'])
for col in ['finbert_score','gpt4o_score',
            'finbert_pos_prob','finbert_neg_prob','finbert_neu_prob']:
    if col in sent_df.columns:
        sent_df[col] = pd.to_numeric(sent_df[col], errors='coerce').fillna(0)

# ── Feature columns ────────────────────────────────────────────────────────
EXCLUDE_COLS  = ['Date', 'date', 'Close', 'LogReturn_Close']
NUM_FEAT_COLS = [c for c in train_num_df.columns if c not in EXCLUDE_COLS]
TARGET_COL    = 'LogReturn_Close'
EMB_COLS      = [c for c in emb_df.columns if c.startswith('emb_')]
SENT_COLS     = [c for c in ['finbert_score','gpt4o_score',
                              'finbert_pos_prob','finbert_neg_prob',
                              'finbert_neu_prob'] if c in sent_df.columns]

print(f'\n  Numerical features : {len(NUM_FEAT_COLS)}')
print(f'  Embedding dims     : {len(EMB_COLS)}')
print(f'  Sentiment features : {SENT_COLS}')
print(f'  Train dates        : {train_num_df["date"].iloc[0]} → {train_num_df["date"].iloc[-1]}')
print(f'  Test dates         : {test_num_df["date"].iloc[0]} → {test_num_df["date"].iloc[-1]}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 4 — Merge Features
# ─────────────────────────────────────────────────────────────────────────

def merge_features(num_df, emb_df, sent_df, num_cols, emb_cols, sent_cols, target_col):
    cols_needed = ['date', target_col] + [c for c in num_cols if c != target_col]
    merged = pd.merge(num_df[cols_needed],
                      emb_df[['date'] + emb_cols],
                      on='date', how='left')
    if sent_cols:
        merged = pd.merge(merged, sent_df[['date'] + sent_cols],
                          on='date', how='left')
        merged[sent_cols] = merged[sent_cols].fillna(0)
    merged[emb_cols] = merged[emb_cols].fillna(0)
    merged = merged.dropna(subset=num_cols).reset_index(drop=True)
    return merged

train_merged = merge_features(train_num_df, emb_df, sent_df,
                               NUM_FEAT_COLS, EMB_COLS, SENT_COLS, TARGET_COL)
test_merged  = merge_features(test_num_df,  emb_df, sent_df,
                               NUM_FEAT_COLS, EMB_COLS, SENT_COLS, TARGET_COL)

train_emb_cov = (train_merged[EMB_COLS[0]] != 0).mean() * 100
test_emb_cov  = (test_merged[EMB_COLS[0]]  != 0).mean() * 100

print(f'  Train rows         : {len(train_merged)}')
print(f'  Test rows          : {len(test_merged)}')
print(f'  Train emb coverage : {train_emb_cov:.1f}%')
print(f'  Test emb coverage  : {test_emb_cov:.1f}%')
print(f'  Target stats       : mean={train_merged[TARGET_COL].mean():.4f}  std={train_merged[TARGET_COL].std():.4f}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 5 — Build Dual-Sequence Dataset
#
# Each sample contains TWO sequences:
#   short_seq: last SHORT_SEQ_LEN days  (for short-term TCN)
#   long_seq : last LONG_SEQ_LEN days   (for long-term TCN)
#   txt_vec  : text embedding for the last day
#   target   : next day log return
# ─────────────────────────────────────────────────────────────────────────

class DualSeqDataset(Dataset):
    def __init__(self, df, num_cols, emb_cols, sent_cols,
                 target_col, short_len, long_len):
        self.short_len = short_len
        self.long_len  = long_len
        self.num_data  = df[num_cols].values.astype(np.float32)
        all_txt        = emb_cols + sent_cols
        self.txt_data  = df[all_txt].values.astype(np.float32)
        self.targets   = df[target_col].values.astype(np.float32)
        # Must have at least long_len rows before first valid sample
        self.n         = len(df) - long_len

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        # Long sequence: last LONG_SEQ_LEN days
        long_seq  = self.num_data[idx : idx + self.long_len]
        # Short sequence: last SHORT_SEQ_LEN days (tail of long sequence)
        short_seq = self.num_data[idx + self.long_len - self.short_len : idx + self.long_len]
        # Text: last day of the long sequence
        txt_vec   = self.txt_data[idx + self.long_len - 1]
        # Target: next day after the long sequence
        target    = self.targets[idx + self.long_len]
        return (
            torch.tensor(short_seq, dtype=torch.float32),
            torch.tensor(long_seq,  dtype=torch.float32),
            torch.tensor(txt_vec,   dtype=torch.float32),
            torch.tensor(target,    dtype=torch.float32),
        )

train_dataset = DualSeqDataset(
    train_merged, NUM_FEAT_COLS, EMB_COLS, SENT_COLS,
    TARGET_COL, SHORT_SEQ_LEN, LONG_SEQ_LEN
)
test_dataset  = DualSeqDataset(
    test_merged, NUM_FEAT_COLS, EMB_COLS, SENT_COLS,
    TARGET_COL, SHORT_SEQ_LEN, LONG_SEQ_LEN
)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

NUM_FEATURES  = len(NUM_FEAT_COLS)
TXT_DIM       = len(EMB_COLS) + len(SENT_COLS)

print(f'  Train sequences  : {len(train_dataset)}')
print(f'  Test sequences   : {len(test_dataset)}')
print(f'  Num features     : {NUM_FEATURES}')
print(f'  Text vector dim  : {TXT_DIM}')
print(f'  Short seq length : {SHORT_SEQ_LEN} days')
print(f'  Long seq length  : {LONG_SEQ_LEN} days')

sb, lb, tb, yb = next(iter(train_loader))
print(f'\n  Batch shapes:')
print(f'    Short seq : {sb.shape}  (batch, short_len, num_features)')
print(f'    Long seq  : {lb.shape}  (batch, long_len, num_features)')
print(f'    Text vec  : {tb.shape}  (batch, txt_dim)')
print(f'    Target    : {yb.shape}  (batch,)')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 6 — Define Dual TCN Model
# ─────────────────────────────────────────────────────────────────────────

class TCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout):
        super().__init__()
        padding   = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size,
                              dilation=dilation, padding=padding)
        self.bn   = nn.BatchNorm1d(out_ch)
        self.gelu = nn.GELU()
        self.drop = nn.Dropout(dropout)
        self.res  = nn.Conv1d(in_ch, out_ch, 1) \
                    if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        out = self.conv(x)[:, :, :x.size(2)]
        return self.drop(self.gelu(self.bn(out))) + self.res(x)


class ResBlock(nn.Module):
    def __init__(self, in_dim, out_dim, dropout):
        super().__init__()
        self.fc   = nn.Linear(in_dim, out_dim)
        self.bn   = nn.BatchNorm1d(out_dim)
        self.gelu = nn.GELU()
        self.drop = nn.Dropout(dropout)
        self.res  = nn.Linear(in_dim, out_dim) \
                    if in_dim != out_dim else nn.Identity()

    def forward(self, x):
        return self.drop(self.gelu(self.bn(self.fc(x)))) + self.res(x)


class TCNBranch(nn.Module):
    """A single TCN branch with input projection and stacked TCN blocks."""
    def __init__(self, num_features, channels, kernel_size, dilations, dropout):
        super().__init__()
        self.input_proj = nn.Linear(num_features, channels)
        self.blocks     = nn.ModuleList([
            TCNBlock(channels, channels, kernel_size, d, dropout)
            for d in dilations
        ])

    def forward(self, x):
        # x: (B, seq_len, num_features)
        x = self.input_proj(x)       # (B, seq_len, channels)
        x = x.permute(0, 2, 1)       # (B, channels, seq_len)
        for block in self.blocks:
            x = block(x)
        return x[:, :, -1]           # last timestep: (B, channels)


class DualTCNPredictor(nn.Module):
    def __init__(self, num_features, txt_dim, tcn_channels, kernel_size,
                 short_dilations, long_dilations, tcn_dropout,
                 text_proj_dim, fusion_dims, dropout):
        super().__init__()

        # ── Short-term TCN branch ─────────────────────────────────────────
        self.short_tcn = TCNBranch(
            num_features, tcn_channels, kernel_size, short_dilations, tcn_dropout
        )

        # ── Long-term TCN branch ──────────────────────────────────────────
        self.long_tcn  = TCNBranch(
            num_features, tcn_channels, kernel_size, long_dilations, tcn_dropout
        )

        # ── Textual branch ────────────────────────────────────────────────
        self.txt_proj  = nn.Sequential(
            nn.Linear(txt_dim, text_proj_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(text_proj_dim * 2, text_proj_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # ── Deep Fusion FNN ───────────────────────────────────────────────
        # Input: short(tcn_ch) + long(tcn_ch) + text(text_proj_dim)
        fusion_input = tcn_channels + tcn_channels + text_proj_dim
        fnn_layers   = []
        in_dim       = fusion_input
        for out_dim in fusion_dims:
            fnn_layers.append(ResBlock(in_dim, out_dim, dropout))
            in_dim = out_dim
        fnn_layers.append(nn.Linear(in_dim, 1))
        self.fusion_fnn = nn.Sequential(*fnn_layers)

    def forward(self, short_seq, long_seq, txt_vec):
        s = self.short_tcn(short_seq)   # (B, tcn_channels)
        l = self.long_tcn(long_seq)     # (B, tcn_channels)
        t = self.txt_proj(txt_vec)      # (B, text_proj_dim)
        fused = torch.cat([s, l, t], dim=1)  # (B, 2*tcn_ch + text_proj_dim)
        return self.fusion_fnn(fused).squeeze(-1)  # (B,)


model = DualTCNPredictor(
    num_features   = NUM_FEATURES,
    txt_dim        = TXT_DIM,
    tcn_channels   = TCN_CHANNELS,
    kernel_size    = TCN_KERNEL_SIZE,
    short_dilations= SHORT_DILATIONS,
    long_dilations = LONG_DILATIONS,
    tcn_dropout    = TCN_DROPOUT,
    text_proj_dim  = TEXT_PROJ_DIM,
    fusion_dims    = FUSION_DIMS,
    dropout        = DROPOUT,
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'  Dual TCN Predictor')
print(f'  Total parameters : {total_params:,}')
print(model)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 7 — Train
# ─────────────────────────────────────────────────────────────────────────

criterion  = nn.HuberLoss(delta=HUBER_DELTA)
optimizer  = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return float(epoch + 1) / float(WARMUP_EPOCHS)
    progress = (epoch - WARMUP_EPOCHS) / max(EPOCHS - WARMUP_EPOCHS, 1)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler      = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
best_val_loss  = float('inf')
patience_count = 0
best_epoch     = 0
train_losses   = []
val_losses     = []

print('=' * 60)
print('  TRAINING — Dual TCN (Short + Long + Text)')
print('=' * 60)

for epoch in range(1, EPOCHS + 1):

    model.train()
    train_loss = 0.0
    for short_seq, long_seq, txt_vec, target in train_loader:
        short_seq = short_seq.to(device)
        long_seq  = long_seq.to(device)
        txt_vec   = txt_vec.to(device)
        target    = target.to(device)
        optimizer.zero_grad()
        pred   = model(short_seq, long_seq, txt_vec)
        loss   = criterion(pred, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for short_seq, long_seq, txt_vec, target in test_loader:
            short_seq = short_seq.to(device)
            long_seq  = long_seq.to(device)
            txt_vec   = txt_vec.to(device)
            target    = target.to(device)
            pred      = model(short_seq, long_seq, txt_vec)
            val_loss += criterion(pred, target).item()
    val_loss /= len(test_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    scheduler.step()

    if epoch % 10 == 0 or epoch == 1:
        lr = optimizer.param_groups[0]['lr']
        print(f'  Epoch {epoch:>3}/{EPOCHS}  '
              f'Train={train_loss:.6f}  '
              f'Val={val_loss:.6f}  '
              f'LR={lr:.2e}')

    if val_loss < best_val_loss:
        best_val_loss  = val_loss
        best_epoch     = epoch
        patience_count = 0
        torch.save(model.state_dict(), 'step9_dualtcn_weights.pth')
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f'\n  Early stopping at epoch {epoch}')
            print(f'  Best epoch    : {best_epoch}')
            print(f'  Best val loss : {best_val_loss:.6f}')
            break

print(f'\n✅ Training complete')
print(f'  Best epoch    : {best_epoch}')
print(f'  Best val loss : {best_val_loss:.6f}')
model.load_state_dict(
    torch.load('step9_dualtcn_weights.pth', map_location=device)
)
print('  Best weights loaded')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 8 — Evaluate
# ─────────────────────────────────────────────────────────────────────────

model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for short_seq, long_seq, txt_vec, target in test_loader:
        pred = model(
            short_seq.to(device),
            long_seq.to(device),
            txt_vec.to(device)
        ).cpu().numpy()
        all_preds.extend(pred)
        all_targets.extend(target.numpy())

preds   = np.array(all_preds)
actuals = np.array(all_targets)

mae     = mean_absolute_error(actuals, preds)
rmse    = np.sqrt(mean_squared_error(actuals, preds))
r2      = r2_score(actuals, preds)
dir_acc = np.mean(np.sign(actuals) == np.sign(preds)) * 100

print('=' * 65)
print('  EVALUATION — Dual TCN (Short + Long + Text)')
print('=' * 65)
print(f'  MAE                  : {mae:.6f}')
print(f'  RMSE                 : {rmse:.6f}')
print(f'  R²                   : {r2:.4f}')
print(f'  Directional Accuracy : {dir_acc:.2f}%')
print()
print('  ─────────────────────────────────────────────────')
print('  FULL COMPARISON TABLE:')
print('  ─────────────────────────────────────────────────')
print(f'  Naive baseline                     : ~51.00%')
print(f'  Baseline LSTM (OHLC only, 11yr)    :  50.94%')
print(f'  Transformer + FinBERT              :  57.35%')
print(f'  TCN + Deep FNN (best so far)       :  58.09%')
print(f'  TCN + Cross-Attention              :  52.94%')
print(f'  TCN + Rolling Sentiment            :  52.94%')
print(f'  Dual TCN (this run)                :  {dir_acc:.2f}%')
print()
if dir_acc > 58.09:
    print(f'  ✅ New best! Dual TCN improved over single TCN by {dir_acc-58.09:+.2f}%')
elif dir_acc > 57.35:
    print(f'  ✅ Better than Transformer but below single TCN ({dir_acc-58.09:+.2f}%)')
else:
    print(f'  ⚠️  Did not beat single TCN ({dir_acc-58.09:+.2f}% vs best)')

metrics_df = pd.DataFrame([{
    'model':                'Dual TCN (Short + Long) + Deep FNN',
    'MAE':                  round(mae, 6),
    'RMSE':                 round(rmse, 6),
    'R2':                   round(r2, 4),
    'Directional_Accuracy': round(dir_acc, 2),
    'Best_Epoch':           best_epoch,
    'Best_Val_Loss':        round(best_val_loss, 6),
}])
metrics_df.to_csv('step9_dualtcn_metrics.csv', index=False)
print(f'\n  Metrics saved → step9_dualtcn_metrics.csv')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 9 — Save and Download
# ─────────────────────────────────────────────────────────────────────────

test_dates = test_merged['date'].values[LONG_SEQ_LEN:]
test_dates = test_dates[:len(preds)]

pred_df = pd.DataFrame({
    'date':             [pd.to_datetime(d).strftime('%d-%b-%Y') for d in test_dates],
    'actual_log_return': actuals,
    'pred_log_return':   preds,
    'actual_direction':  np.where(actuals >= 0, 'UP', 'DOWN'),
    'pred_direction':    np.where(preds   >= 0, 'UP', 'DOWN'),
    'direction_correct': np.sign(actuals) == np.sign(preds),
    'error':             np.abs(actuals - preds),
})
pred_df.to_csv('step9_dualtcn_predictions.csv', index=False)

loss_df = pd.DataFrame({
    'epoch':      list(range(1, len(train_losses)+1)),
    'train_loss': train_losses,
    'val_loss':   val_losses,
})
loss_df.to_csv('step9_dualtcn_loss.csv', index=False)

output_files = [
    ('step9_dualtcn_predictions.csv', 'Predicted vs actual'),
    ('step9_dualtcn_metrics.csv',     'Evaluation metrics'),
    ('step9_dualtcn_loss.csv',        'Training loss curve'),
    ('step9_dualtcn_weights.pth',     'Model weights'),
]

print('=' * 60)
print('  DUAL TCN COMPLETE')
print('=' * 60)
for fname, desc in output_files:
    if os.path.exists(fname):
        print(f'  ✓ {fname}  ({os.path.getsize(fname)/1024:.1f} KB)')

print(f'\n  Directional Accuracy : {dir_acc:.2f}%')
print(f'  Best epoch           : {best_epoch}')

print('\n⬇️  Downloading...')
for fname, _ in output_files:
    if os.path.exists(fname):
        files.download(fname)
        print(f'  ✓ {fname}')
print('\n✅ Done!')